# Reuters Oil & Energy News Crawler
<!-- Created: 2026-03-23 -->

| 항목 | 내용 |
|------|------|
| 수집 방식 | Google News RSS (`site:reuters.com`) |
| 필드 | date · headline · summary · url · keyword · source |
| 카테고리 | supply · demand · geopolitical · financial |
| 출력 파일 | `oil_{카테고리}_{MM}.{YYYY}.csv / .xlsx` |

> **참고** Reuters 사이트는 DataDome 봇 차단으로 직접 크롤링 불가.  
> `url`은 Google News RSS 링크이며, 클릭 시 Reuters 원문으로 이동합니다.  
> `summary`는 Google News RSS의 description 필드에서 추출합니다.

In [1]:
# import sys
# !{sys.executable} -m pip install openpyxl

In [2]:
# import sys
# !{sys.executable} -m pip install feedparser

In [3]:
import re
import time
import random
import calendar
from datetime import datetime, date, timezone, timedelta

import feedparser
import pandas as pd

print("Import 완료")

Import 완료


## 1. 설정

In [4]:
# ── 수집 기간: 이 네 줄만 수정 ──────────────────────
START_YEAR  = 2007
START_MONTH = 8
END_YEAR    = 2015
END_MONTH   = 12
# ──────────────────────────────────────────────────

START_DATE = f"{START_YEAR}-{START_MONTH:02d}-01"
END_DATE   = f"{END_YEAR}-{END_MONTH:02d}-{calendar.monthrange(END_YEAR, END_MONTH)[1]:02d}"
RANGE_STR  = f"{START_MONTH:02d}.{str(START_YEAR)[2:]}-{END_MONTH:02d}.{str(END_YEAR)[2:]}"  # 예: 01.26-03.26

CATEGORY_KEYWORDS = {
    "supply": [
        "crude", "opec", "supply", "inventory", "production",
    ],
    "demand": [
        "oil", "energy", "demand",
    ],
    "geopolitical": [
        "middle east", "russia", "ukraine", "iran",
        "sanction", "war", "conflict",
    ],
    "financial": [
        "brent", "wti",
    ],
}

print(f"수집 기간: {START_DATE} ~ {END_DATE}  (파일명: {RANGE_STR})")
for cat, kws in CATEGORY_KEYWORDS.items():
    print(f"  [{cat}] {kws}")

수집 기간: 2007-08-01 ~ 2015-12-31  (파일명: 08.07-12.15)
  [supply] ['crude', 'opec', 'supply', 'inventory', 'production']
  [demand] ['oil', 'energy', 'demand']
  [geopolitical] ['middle east', 'russia', 'ukraine', 'iran', 'sanction', 'war', 'conflict']
  [financial] ['brent', 'wti']


## 2. 수집 함수

In [5]:
def fetch_rss(keyword: str, start: str, end: str) -> list[dict]:
    """Google News RSS로 site:reuters.com + keyword 기사 수집 (월 단위 호출용)."""
    next_day = date.fromisoformat(end) + timedelta(days=1)
    q = f'site:reuters.com+{keyword.replace(" ", "+")}+after:{start}+before:{next_day}'
    feed = feedparser.parse(
        f'https://news.google.com/rss/search?q={q}&hl=en-US&gl=US&ceid=US:en'
    )
    start_d = date.fromisoformat(start)
    end_d   = date.fromisoformat(end)

    results = []
    for entry in feed.entries:
        try:
            pub = datetime(*entry.published_parsed[:6], tzinfo=timezone.utc).date()
        except Exception:
            continue
        if not (start_d <= pub <= end_d):
            continue

        headline = (
            entry.title
            .removesuffix(" - Reuters")
            .removesuffix(" | Reuters")
            .strip()
        )

        results.append({
            "date":     pub.isoformat(),
            "headline": headline,
            "url":      entry.link,
            "keyword":  keyword,
            "source":   "Reuters",
        })
    return results


def iter_months(sy: int, sm: int, ey: int, em: int):
    """(START_YEAR, START_MONTH) ~ (END_YEAR, END_MONTH) 월 목록 반환."""
    y, m = sy, sm
    while (y, m) <= (ey, em):
        yield y, m
        m += 1
        if m > 12:
            m, y = 1, y + 1


print("fetch_rss() / iter_months() 함수 정의 완료")

fetch_rss() / iter_months() 함수 정의 완료


## 3. 카테고리별 수집

In [6]:
category_rows: dict[str, list[dict]] = {cat: [] for cat in CATEGORY_KEYWORDS}

months = list(iter_months(START_YEAR, START_MONTH, END_YEAR, END_MONTH))

for cat, keywords in CATEGORY_KEYWORDS.items():
    print(f"\n[{cat}]")
    for kw in keywords:
        kw_total = 0
        for y, m in months:
            ms = f"{y}-{m:02d}-01"
            me = f"{y}-{m:02d}-{calendar.monthrange(y, m)[1]:02d}"
            items = fetch_rss(kw, ms, me)
            category_rows[cat].extend(items)
            kw_total += len(items)
            time.sleep(random.uniform(0.4, 0.9))
        print(f"  '{kw}' → {kw_total}건")

print("\n── 합계 (중복 포함) ──")
for cat, rows in category_rows.items():
    print(f"  [{cat}] {len(rows)}건")


[supply]
  'crude' → 3490건
  'opec' → 359건
  'supply' → 9557건
  'inventory' → 558건
  'production' → 9282건

[demand]
  'oil' → 9622건
  'energy' → 9590건
  'demand' → 9484건

[geopolitical]
  'middle east' → 9581건
  'russia' → 9645건
  'ukraine' → 9692건
  'iran' → 8698건
  'sanction' → 3833건
  'war' → 9542건
  'conflict' → 9301건

[financial]
  'brent' → 836건
  'wti' → 71건

── 합계 (중복 포함) ──
  [supply] 23246건
  [demand] 28696건
  [geopolitical] 60292건
  [financial] 907건


## 4. 정제 & 저장

In [ ]:
NOISE_PATTERN = r'Stock Price|Latest News|\s\|\s'
COLS = ["date", "headline", "url", "keyword", "source"]

category_dfs: dict[str, pd.DataFrame] = {}

for cat, rows in category_rows.items():
    df = pd.DataFrame(rows)
    if df.empty:
        print(f"[{cat}] 수집 결과 없음, 건너뜀")
        category_dfs[cat] = df
        continue

    # ① 제목 기준 중복 제거
    df.drop_duplicates(subset="url", inplace=True)

    # ② 비뉴스 항목 제거 (주가 페이지 등)
    df = df[~df["headline"].str.contains(NOISE_PATTERN, regex=True, na=False)]

    # ③ 날짜 내림차순 정렬
    df.sort_values("date", ascending=False, inplace=True)
    df.reset_index(drop=True, inplace=True)

    df = df[COLS]

    # ④ 저장
    csv_path = f"../data/Crawling_raw/oil_{cat}_{RANGE_STR}.csv"
    df.to_csv(csv_path, index=False, encoding="utf-8-sig")
    print(f"[{cat}] 정제 후 {len(df)}건 → {csv_path}")

    category_dfs[cat] = df

## 5. 통계

In [8]:
for cat, df in category_dfs.items():
    if df.empty:
        print(f"[{cat}] 데이터 없음\n")
        continue
    print(f"\n=== [{cat}] 키워드별 기사 수 ===")
    print(df["keyword"].value_counts().to_string())
    print(f"\n=== [{cat}] 날짜별 기사 수 ===")
    print(df.groupby("date").size().to_string())


=== [supply] 키워드별 기사 수 ===
keyword
supply        8661
production    6375
crude         3489
inventory      326
opec           208

=== [supply] 날짜별 기사 수 ===
date
2007-08-09    293
2007-08-12      1
2007-08-13      3
2007-08-14      3
2007-08-15      1
2007-08-16      1
2007-08-19      1
2007-08-20      1
2007-08-21      2
2007-08-22      2
2007-08-23      4
2007-08-24      1
2007-08-27      1
2007-08-28      1
2007-08-31      1
2007-09-02      1
2007-09-03      3
2007-09-04      6
2007-09-05     10
2007-09-06      1
2007-09-07      6
2007-09-08      1
2007-09-09      1
2007-09-10      2
2007-09-11      7
2007-09-12     10
2007-09-13      9
2007-09-14      4
2007-09-16      5
2007-09-17     11
2007-09-18      4
2007-09-19     10
2007-09-20      3
2007-09-21      5
2007-09-22      1
2007-09-23      1
2007-09-24      1
2007-09-25      8
2007-09-26      9
2007-09-27     13
2007-09-28      3
2007-09-30      1
2007-10-01      4
2007-10-02      3
2007-10-03      5
2007-10-04     12
2007-10-0